# 🚗 Train Vehicle Detection Model

> **Đề tài:** Hệ thống Giám sát Phương tiện Giao thông – DATN_VTHUW  
> **Model:** YOLOv8s – Phát hiện xe hơi, xe máy, xe tải, xe buýt, xe đạp  
> **Dataset:** Roboflow Universe (Vehicle Detection, 14,000+ ảnh)  
> **Output:** `vehicle_detection.pt` → lưu Google Drive

⚡ **Bật GPU:** Runtime → Change runtime type → **T4 GPU**

---
### Danh sách dataset có thể dùng:
| Dataset | Ảnh | Classes | Link |
|---------|-----|---------|------|
| Vehicle Detection (Roboflow) | 14,000+ | car, motorcycle, truck, bus | [Link](https://universe.roboflow.com/vehicle-detection-3mmwj/vehicle-detection-kogvf) |
| COCO (pretrained) | 118,000 | 80 classes (bao gồm xe) | YOLOv8 pretrained |
| Vietnam Traffic (tự gán nhãn) | tùy | xe, người | Upload lên Drive |

## 📦 Bước 1: Cài đặt & kiểm tra GPU

In [ ]:
# Kiểm tra GPU
!nvidia-smi
import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Cài thư viện
!pip install ultralytics roboflow --quiet

import ultralytics
ultralytics.checks()
print('\n✅ Ultralytics sẵn sàng')

## ☁️ Bước 2: Kết nối Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Tạo thư mục lưu trữ trên Drive
DRIVE_DIR = '/content/drive/MyDrive/DATN_TrafficAI'
MODEL_DIR = f'{DRIVE_DIR}/models'
LOG_DIR   = f'{DRIVE_DIR}/logs'

for d in [MODEL_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'📁 Models sẽ lưu tại: {MODEL_DIR}')
print(f'📁 Logs sẽ lưu tại:   {LOG_DIR}')

## 📂 Bước 3: Tải Dataset

**Chọn 1 trong 3 cách:**

In [ ]:
# ==============================================================
# 🅰️ CÁCH A: Roboflow API (khuyến nghị – dataset chất lượng cao)
# 1. Đăng ký tại https://app.roboflow.com
# 2. Lấy API key tại Settings > API Keys
# 3. Dán vào ô RF_API_KEY bên dưới
# ==============================================================

USE_ROBOFLOW = True  # Đổi thành False nếu dùng cách B hoặc C

RF_API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # <-- THAY VÀO ĐÂY

# Dataset options (uncomment 1 trong các dòng bên dưới):
# --- Option 1: Vehicle Detection tổng hợp (14k ảnh) ---
RF_WORKSPACE = 'vehicle-detection-3mmwj'
RF_PROJECT   = 'vehicle-detection-kogvf'
RF_VERSION   = 8

# --- Option 2: Traffic vehicles (Vietnam style) ---
# RF_WORKSPACE = 'traffic-1vkjj'
# RF_PROJECT   = 'vehicle-detection-auqd9'
# RF_VERSION   = 2

if USE_ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key=RF_API_KEY)
    project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    dataset = project.version(RF_VERSION).download('yolov8')
    DATASET_PATH = dataset.location
    print(f'✅ Dataset tải về: {DATASET_PATH}')

In [ ]:
# ==============================================================
# 🅱️ CÁCH B: COCO128 – Dataset mẫu sẵn có (không cần API key)
# Nhỏ hơn nhưng đủ để test pipeline, bao gồm xe cộ
# ==============================================================

# Bỏ comment để dùng:
# import os
# os.makedirs('/content/datasets', exist_ok=True)
# !cd /content/datasets && \
#   wget -q https://ultralytics.com/assets/coco128.zip && \
#   unzip -q coco128.zip
# DATASET_PATH = '/content/datasets/coco128'

# Dùng data.yaml của COCO128:
# !wget -q -O /content/coco128.yaml https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco128.yaml
# YAML_FILE = '/content/coco128.yaml'

In [ ]:
# ==============================================================
# 🅲 CÁCH C: Upload dataset tự có lên Google Drive
# Cấu trúc cần có:
#   MyDrive/DATN_TrafficAI/datasets/vehicle/
#     images/train/, images/val/
#     labels/train/, labels/val/
#     data.yaml
# ==============================================================

# Bỏ comment để dùng:
# DATASET_PATH = '/content/drive/MyDrive/DATN_TrafficAI/datasets/vehicle'
# import glob
# YAML_FILE = glob.glob(f'{DATASET_PATH}/*.yaml')[0]
# print(f'Dùng dataset tự có: {YAML_FILE}')

In [ ]:
# Tìm file data.yaml
import glob, os

try:
    yaml_files = glob.glob(f'{DATASET_PATH}/**/*.yaml', recursive=True)
    YAML_FILE = yaml_files[0] if yaml_files else None
    print(f'📄 Data config: {YAML_FILE}')

    # Hiển thị nội dung yaml
    with open(YAML_FILE) as f:
        content = f.read()
    print('\n--- Nội dung data.yaml ---')
    print(content)

    # Đếm số ảnh
    train_imgs = len(glob.glob(f'{DATASET_PATH}/train/images/*'))
    val_imgs   = len(glob.glob(f'{DATASET_PATH}/valid/images/*'))
    print(f'\nTrain: {train_imgs} ảnh | Val: {val_imgs} ảnh')
except Exception as e:
    print(f'⚠️  {e} – Hãy tải dataset trước ở bước trên')

## 🚀 Bước 4: Huấn luyện YOLOv8

| Tham số | Giá trị | Giải thích |
|---------|---------|------------|
| `model` | yolov8s.pt | Small model – cân bằng tốc độ & chính xác |
| `epochs` | 50 | Số lần lặp qua dataset |
| `imgsz` | 640 | Kích thước ảnh |
| `batch` | 16 | Batch size (phù hợp T4 16GB) |
| `patience` | 15 | Dừng sớm nếu không cải thiện |
| `amp` | True | Mixed precision → nhanh 2x |

In [ ]:
from ultralytics import YOLO
import torch

# Chọn model size (uncomment 1 dòng):
# MODEL = 'yolov8n.pt'  # Nano  – nhanh nhất, ít chính xác nhất (phù hợp CPU)
MODEL = 'yolov8s.pt'    # Small – khuyến nghị cho T4 Colab
# MODEL = 'yolov8m.pt'  # Medium – chính xác hơn, cần nhiều VRAM hơn

model = YOLO(MODEL)
print(f'✅ Loaded: {MODEL}')
print(f'   Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Bắt đầu training
results = model.train(
    data=YAML_FILE,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    name='vehicle_detection',
    project='/content/runs',
    save=True,
    save_period=10,       # Lưu checkpoint mỗi 10 epoch
    exist_ok=True,
    amp=True,             # Mixed precision
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    # Augmentation
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    degrees=5.0,          # Xoay nhẹ ±5 độ
    translate=0.1,        # Dịch chuyển 10%
    scale=0.5,            # Scale 50-150%
    hsv_h=0.015,          # Hue augmentation
    hsv_s=0.7,            # Saturation
    hsv_v=0.4,            # Value/brightness
)

print('\n' + '='*60)
print('✅ Training hoàn tất!')
print(f'   mAP50: {results.results_dict.get("metrics/mAP50(B)", 0):.4f}')
print('='*60)

## 📊 Bước 5: Đánh giá Model

In [ ]:
from IPython.display import Image as IPImage, display
import glob, os

run_dir = '/content/runs/vehicle_detection'

# Hiển thị charts
for img_name in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    path = f'{run_dir}/{img_name}'
    if os.path.exists(path):
        print(f'\n--- {img_name} ---')
        display(IPImage(path, width=800))

In [ ]:
# Validate trên tập test
best_model = YOLO(f'{run_dir}/weights/best.pt')
metrics = best_model.val()

print('\n=== KẾT QUẢ ĐÁNH GIÁ ===')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## 💾 Bước 6: Lưu Model lên Google Drive

In [ ]:
import shutil

best_pt = f'/content/runs/vehicle_detection/weights/best.pt'
dest    = f'{MODEL_DIR}/vehicle_detection.pt'

shutil.copy2(best_pt, dest)
print(f'✅ Model lưu tại: {dest}')
print(f'   Kích thước: {os.path.getsize(dest)/1e6:.1f} MB')

# Lưu cả kết quả training
shutil.copytree('/content/runs/vehicle_detection', f'{LOG_DIR}/vehicle_detection', dirs_exist_ok=True)
print(f'✅ Logs lưu tại: {LOG_DIR}/vehicle_detection')

print('\n🎉 Hoàn tất! Tải file về máy:')
print(f'   Google Drive → DATN_TrafficAI/models/vehicle_detection.pt')
print(f'   Copy vào: DATN_VTHUW/models/vehicle_detection.pt')

In [ ]:
# (Tùy chọn) Tải thẳng về máy từ Colab
from google.colab import files

# Bỏ comment để download:
# files.download(f'/content/runs/vehicle_detection/weights/best.pt')